# Random Forest Regression — Original Dataset

## Final Results

The tuned Random Forest model was evaluated on the unseen test set.

- **MAE:** 0.883
- **RMSE:** 1.430
- **R²:** 0.797

Best hyperparameters selected by 3-fold cross-validation:

- `n_estimators = 500`
- `max_depth = None`
- `max_features = 0.7`
- `min_samples_leaf = 1`

Best cross-validation MAE: **1.030**

For comparison, the mean DummyRegressor baseline gave:

- **MAE:** 2.358
- **RMSE:** 3.173
- **R²:** ~0

---

## Modeling Workflow

1. Load the extracted ENSDF dataset  
   - 3371 isotopes, 20 columns

2. Select isotopes with finite half-lives  
   - 2796 samples retained

3. Define the regression target  
   - `log_half_life = log10(half_life_seconds)`

4. Select six input features  
   - `Z`
   - `N`
   - `Q_beta_keV`
   - `S_n_keV`
   - `S_p_keV`
   - `Q_alpha_keV`

5. Retain missing feature values  
   - No imputation or row removal was applied
   - Random Forest in scikit-learn 1.7.2 handles missing values natively

6. Split the dataset into training and test sets  
   - 80% training: 2236 samples
   - 20% test: 560 samples
   - `random_state = 1`

7. Establish a baseline  
   - Mean `DummyRegressor`

8. Train an initial Random Forest regression model  
   - Tutorial-based initial settings: `n_estimators=100`, `max_features=0.5`, `min_samples_leaf=3`

9. Perform 3-fold cross-validation using only the training set

10. Perform initial hyperparameter grid search  
    - 36 parameter combinations
    - Best: `n_estimators=300`, `max_depth=None`, `max_features=0.5`, `min_samples_leaf=1`
    - CV MAE = 1.037

11. Refine the hyperparameter search  
    - 24 additional combinations
    - Best: `n_estimators=500`, `max_depth=None`, `max_features=0.7`, `min_samples_leaf=1`
    - CV MAE = 1.030

12. Select the best model and evaluate it once on the unseen test set

In [1]:
import sklearn
from sklearn.ensemble import RandomForestRegressor

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.7.2


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.ensemble import RandomForestRegressor

RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

In [ ]:
df = pd.read_csv(
    "../ensdf_ground_state_candidates.csv",
    keep_default_na=False,
    na_values=[""]
)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (3371, 20)


,nuclide,element,Z,A,N,dataset_name,ground_energy_keV,spin_parity_raw,half_life_raw,half_life_value,half_life_unit,half_life_seconds,half_life_status,Q_beta_keV,S_n_keV,S_p_keV,Q_alpha_keV,ground_state_ambiguous,ground_state_candidate_count,source_file
0,1H,H,1,1,0,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,NaN,NaN,False,1,ensdf.001
1,2H,H,1,2,1,ADOPTED LEVELS,0.0,1+,STABLE,NaN,NaN,NaN,stable,NaN,2224.566,2224.566,1.0,False,1,ensdf.002
2,3H,H,1,3,2,ADOPTED LEVELS,0.0,1/2+,12.32 Y,12.32,Y,388789632.0,finite,18.5906,6257.233,NaN,NaN,False,1,ensdf.003
3,3HE,HE,2,3,1,ADOPTED LEVELS,0.0,1/2+,STABLE,NaN,NaN,NaN,stable,NaN,NaN,5493.477,NaN,False,1,ensdf.003
4,4H,H,1,4,3,ADOPTED LEVELS,0.0,2-,NaN,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,False,1,ensdf.004


In [4]:
reg_df = df[df["half_life_status"] == "finite"].copy()

print("Finite half-life samples:", len(reg_df))

Finite half-life samples: 2796


In [5]:
reg_df["log_half_life"] = np.log10(
    reg_df["half_life_seconds"]
)

display(
    reg_df[
        ["nuclide", "half_life_seconds", "log_half_life"]
    ].head()
)

,nuclide,half_life_seconds,log_half_life
2,3H,3.887896e+08,8.589715
11,6HE,8.067000e-01,-0.093288
17,7BE,4.598208e+06,6.662589
19,8HE,1.191000e-01,-0.924088
20,8LI,8.399000e-01,-0.075772


In [6]:
FEATURES = [
    "Z",
    "N",
    "Q_beta_keV",
    "S_n_keV",
    "S_p_keV",
    "Q_alpha_keV"
]

TARGET = "log_half_life"

X = reg_df[FEATURES].copy()
y = reg_df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2796, 6)
y shape: (2796,)


In [7]:
missing_summary = pd.DataFrame({
    "Missing count": X.isna().sum(),
    "Missing (%)": X.isna().mean() * 100
})

display(missing_summary)

,Missing count,Missing (%)
Z,0,0.000000
N,0,0.000000
Q_beta_keV,826,29.542203
S_n_keV,624,22.317597
S_p_keV,630,22.532189
Q_alpha_keV,615,21.995708


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_SEED
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

Training samples: 2236
Test samples: 560


In [9]:
dummy_model = DummyRegressor(strategy="mean")

dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_test)

In [10]:
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

dummy_results = evaluate_regression(
    y_test,
    dummy_pred
)

dummy_results

{'MAE': 2.3584085725340986,
 'RMSE': 3.1726366097213416,
 'R2': -2.792962555164813e-05}

In [11]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_features=0.5,
    min_samples_leaf=3,
    random_state=RANDOM_SEED,
    n_jobs=2
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,0.5
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
from sklearn.model_selection import KFold

rf_cv = list(
    KFold(
        n_splits=3,
        shuffle=True,
        random_state=RANDOM_SEED
    ).split(X_train)
)

print("Number of CV folds:", len(rf_cv))

Number of CV folds: 3


In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [10, 20, None],
    "max_features": [0.5, 1.0],
    "min_samples_leaf": [1, 3]
}

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=2
    ),
    param_grid=param_grid,
    cv=rf_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
    verbose=1
)

In [14]:
grid_rf.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


,estimator,RandomForestR...andom_state=1)
,param_grid,"{'max_depth': [10, 20, ...], 'max_features': [0.5, 1.0], 'min_samples_leaf': [1, 3], 'n_estimators': [100, 300, ...]}"
,scoring,'neg_mean_absolute_error'
,n_jobs,2
,refit,True
,cv,"[(array([ 1, ..., 2233, 2235]), ...), (array([ 0, ..., 2234, 2235]), ...), ...]"
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [15]:
print("Best parameters:")
print(grid_rf.best_params_)

print("Best CV MAE:")
print(-grid_rf.best_score_)

Best parameters:
{'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 300}
Best CV MAE:
1.0368744754004788


In [16]:
param_grid_refined = {
    "n_estimators": [300, 500],
    "max_depth": [20, None],
    "max_features": [0.3, 0.5, 0.7],
    "min_samples_leaf": [1, 2]
}

grid_rf_refined = GridSearchCV(
    estimator=RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=2
    ),
    param_grid=param_grid_refined,
    cv=rf_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
    verbose=1
)

In [17]:
grid_rf_refined.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


,estimator,RandomForestR...andom_state=1)
,param_grid,"{'max_depth': [20, None], 'max_features': [0.3, 0.5, ...], 'min_samples_leaf': [1, 2], 'n_estimators': [300, 500]}"
,scoring,'neg_mean_absolute_error'
,n_jobs,2
,refit,True
,cv,"[(array([ 1, ..., 2233, 2235]), ...), (array([ 0, ..., 2234, 2235]), ...), ...]"
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,500


In [18]:
print("Best refined parameters:")
print(grid_rf_refined.best_params_)

print("Best refined CV MAE:")
print(-grid_rf_refined.best_score_)

Best refined parameters:
{'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 1, 'n_estimators': 500}
Best refined CV MAE:
1.0299659272928674


In [19]:
best_rf = grid_rf_refined.best_estimator_

best_rf

,n_estimators,500
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,0.7
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [20]:
# Predict the unseen test set using the tuned Random Forest model
best_rf_pred = best_rf.predict(X_test)

# Evaluate final test performance
best_rf_results = evaluate_regression(
    y_test,
    best_rf_pred
)

best_rf_results

{'MAE': 0.8832064687183543,
 'RMSE': 1.430451901345877,
 'R2': 0.7967090780371178}